# Config 4: camera video and lying labels

Watch the camera frames of the Config 4 recordings next to their hand-made pose labels, and see what the classifier is asked to predict: **lying** (lying occurred in the window) against **not lying**. The section "Browse the recordings" lets you cycle through the recordings and step through each one.

Config 4 is the only sensor config with pose labels, and the only one used for model development. Frames are in **capture order** (time order). In two recordings the stored `frame_idx` is not the time order, so the notebook counts positions in time order, as the annotation tool does.


Run `python -m data.bake` first if `data/baked/floor_posture` does not exist.

The camera frames show identifiable people: keep this notebook and its outputs local, and clear the outputs before you commit.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_from_disk
from IPython.display import HTML, display
from matplotlib import animation
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from data.annotations import capture_positions

plt.rcParams["animation.embed_limit"] = 80  # MB; the inline video player embeds every frame

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ds = load_from_disk(repo / "data/baked/floor_posture")

# metadata only: the camera images are read later, one clip at a time
heavy = ["image", "radar_payload", "heatmap", "pointcloud"]
meta = ds.remove_columns(heavy).to_pandas()  # row i of `meta` is row i of `ds`
meta["capture_idx"] = capture_positions(meta)  # the stored frame_idx is not the time order in two recordings
c4 = meta[meta.cfg_file == "Config_4.cfg"].sort_values(["recording", "capture_idx"]).copy()
c4["lying"] = c4.label_pose.fillna("").str.startswith("lying")
c4["person"] = c4.person.replace("NA", "unknown")
print(f"{c4.recording.nunique()} Config 4 recordings, {len(c4)} frames, {int(c4.lying.sum())} lying frames")

## Labels

`label_pose` has one value per frame: `standing`, `moving`, `sitting`, `lying_left`, `lying_right`, or empty (not labelled). The task merges `lying_left` and `lying_right` into **lying** and calls everything else **not lying**.

In [ ]:
c4.label_pose.value_counts(dropna=False).rename("frames").to_frame()

## Recordings

One row per recording. `lying runs` are stretches of consecutive lying frames: lying is annotated in short runs (about 0.2 to 1 s), so a recording has few of them. A frame is about 0.2 s.

In [ ]:
def lying_runs(rec):
    """(first, last) row positions of every run of consecutive lying frames in one recording (sorted by frame)."""
    flags = rec.lying.to_numpy()
    edges = np.diff(np.r_[0, flags.astype(int), 0])
    return list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1) - 1))


runs = {name: lying_runs(rec) for name, rec in c4.groupby("recording")}
overview = c4.groupby("recording").agg(
    person=("person", "first"), room=("room", "first"), frames=("frame_idx", "size"),
    unlabelled=("label_pose", lambda s: int(s.isna().sum())), lying_frames=("lying", "sum"),
)
overview["lying runs"] = pd.Series({k: len(v) for k, v in runs.items()})
overview["seconds"] = (overview.frames * 0.2).round()
overview.sort_values("lying_frames", ascending=False)

## Timeline of every recording

Each strip is one recording, one cell per frame. Lying is red, and grey means not labelled (the last recording has no labels at all). Use it to find the moments worth watching.

In [ ]:
POSES = ["standing", "moving", "sitting", "lying", "not labelled"]
COLOURS = ["#9ecae1", "#c6dbef", "#fdd49e", "#d7301f", "#eeeeee"]


def pose_code(label):
    if label is None or label != label:
        return 4
    return 3 if label.startswith("lying") else POSES.index(label)


names = list(overview.sort_values("lying_frames", ascending=False).index)
longest = int(overview.frames.max())
grid = np.full((len(names), longest), np.nan)
for row, name in enumerate(names):
    codes = c4[c4.recording == name].label_pose.map(pose_code).to_numpy()
    grid[row, : len(codes)] = codes

fig, ax = plt.subplots(figsize=(12, 0.45 * len(names) + 1.2))
ax.imshow(grid, aspect="auto", interpolation="nearest", cmap=ListedColormap(COLOURS), vmin=0, vmax=4)
ax.set_yticks(range(len(names)), [f"{n.split('__')[1]}  {overview.person[n]}" for n in names], fontsize=8)
ax.set_xlabel("frame (0.2 s each)")
ax.legend(handles=[Patch(color=c, label=p) for p, c in zip(POSES, COLOURS)], ncol=5, loc="upper center",
          bbox_to_anchor=(0.5, -0.18), frameon=False, fontsize=8)
plt.show()

## Browse the recordings

Pick a recording, then step through it. **◀ recording / recording ▶** cycle through the recordings, **◀ lying / lying ▶** jump to the previous or next lying run, the **play** button plays the recording, and the slider (click it, then use the arrow keys) steps one frame (0.2 s) at a time. Under the frame, the strip shows the labels of the whole recording, with a black line at the current frame and a red marker above every lying run.

The recordings are in the order of their names, which is the order of recording time. The camera image is read when you move, so nothing is loaded up front.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

POSE_COLOUR = dict(zip(POSES, COLOURS))
recordings = sorted(overview.index)


def frames_of(recording):
    return c4[c4.recording == recording]


def draw_frame(recording, pos):
    """A figure with camera frame number `pos` of one recording, its label, and the label strip of the recording."""
    rec = frames_of(recording)
    row = rec.iloc[pos]
    lying = bool(row.lying)
    label = row.label_pose if isinstance(row.label_pose, str) else "not labelled"
    image = ds[int(rec.index[pos])]["image"].convert("RGB")
    codes = rec.label_pose.map(pose_code).to_numpy()

    fig, (ax_img, ax_strip) = plt.subplots(2, 1, figsize=(6.4, 5.8), dpi=80, gridspec_kw={"height_ratios": [7, 1]})
    ax_img.imshow(image)
    ax_img.set_xticks([]), ax_img.set_yticks([])
    for spine in ax_img.spines.values():
        spine.set_edgecolor("#d7301f" if lying else "#cccccc")
        spine.set_linewidth(5 if lying else 1)
    ax_img.set_title(f"{label}   ({'LYING' if lying else 'not lying'})", fontsize=13, color="#d7301f" if lying else "black")
    ax_img.set_xlabel(f"{recording.split('__')[-1]}  ·  {overview.person[recording]}  ·  frame {pos + 1} of {len(rec)}  ·  {pos * 0.2:.1f} s",
                      fontsize=9)

    ax_strip.imshow(codes[None, :], aspect="auto", interpolation="nearest", cmap=ListedColormap(COLOURS), vmin=0, vmax=4)
    ax_strip.axvline(pos, color="black", lw=2)
    starts = [a for a, _ in runs[recording]]
    ax_strip.scatter(starts, [-0.85] * len(starts), marker="v", color="#d7301f", s=22, clip_on=False)
    ax_strip.set_yticks([])
    ticks = np.arange(0, len(codes), 300)  # every minute
    ax_strip.set_xticks(ticks, [f"{int(t * 0.2 // 60)} min" for t in ticks], fontsize=8)
    ax_strip.legend(handles=[Patch(color=c, label=p) for p, c in zip(POSES, COLOURS)], ncol=5, loc="upper center",
                    bbox_to_anchor=(0.5, -0.9), frameon=False, fontsize=7)
    fig.tight_layout()
    return fig


def option(name):
    lying = int(overview.lying_frames[name])
    state = "not labelled" if overview.unlabelled[name] == overview.frames[name] else f"{lying} lying frames"
    return (f"{name.split('__')[-1]}  ·  {overview.person[name]}  ·  {overview.frames[name] * 0.2 / 60:.1f} min  ·  {state}", name)


recording_menu = widgets.Dropdown(options=[option(n) for n in recordings], value=recordings[0], description="Recording",
                                  layout=widgets.Layout(width="620px"))
prev_recording = widgets.Button(description="◀ recording", layout=widgets.Layout(width="115px"))
next_recording = widgets.Button(description="recording ▶", layout=widgets.Layout(width="115px"))
prev_run = widgets.Button(description="◀ lying", layout=widgets.Layout(width="90px"))
next_run = widgets.Button(description="lying ▶", layout=widgets.Layout(width="90px"))
frame = widgets.IntSlider(value=0, min=0, max=len(frames_of(recordings[0])) - 1, description="Frame",
                          continuous_update=False, layout=widgets.Layout(width="560px"))
play = widgets.Play(value=0, min=0, max=frame.max, step=1, interval=200)  # 200 ms per frame is real time
widgets.jslink((play, "value"), (frame, "value"))
picture = widgets.Output()


def show(*_):
    fig = draw_frame(recording_menu.value, frame.value)
    with picture:
        clear_output(wait=True)
        display(fig)
    plt.close(fig)


def change_recording(*_):
    last = len(frames_of(recording_menu.value)) - 1
    frame.max = play.max = last
    if frame.value != 0:
        frame.value = 0  # this redraws
    else:
        show()


def step_recording(direction):
    recording_menu.value = recordings[(recordings.index(recording_menu.value) + direction) % len(recordings)]


def jump_to_run(direction):
    starts = [a for a, _ in runs[recording_menu.value]]
    if not starts:
        return
    later = [a for a in starts if a > frame.value]
    earlier = [a for a in starts if a < frame.value]
    frame.value = (later[0] if later else starts[0]) if direction > 0 else (earlier[-1] if earlier else starts[-1])


recording_menu.observe(change_recording, "value")
frame.observe(show, "value")
prev_recording.on_click(lambda _: step_recording(-1))
next_recording.on_click(lambda _: step_recording(+1))
prev_run.on_click(lambda _: jump_to_run(-1))
next_run.on_click(lambda _: jump_to_run(+1))

show()
display(widgets.VBox([
    widgets.HBox([prev_recording, recording_menu, next_recording]),
    widgets.HBox([prev_run, next_run, play, frame]),
    picture,
]))

## What the classifier sees: windows

The model predicts one label per **window** of 16 frames (3.2 s), moved forward 8 frames at a time. With the default label rule `any_lying` a window is **lying** if at least one of its frames is lying, and **not lying** if none is. With the rule `last`, a window takes the label of its last frame, which misses most of the short lying runs.

This cell applies both rules to the Config 4 recordings. It repeats the rule from `data/windowing.py`, so it can differ from a real run when frames are missing or unlabelled.

In [ ]:
WINDOW, STRIDE = 16, 8


def windows(rec):
    """(start, end, any_lying, last_is_lying) of every window of one recording."""
    flags = rec.lying.to_numpy()
    return [(s, s + WINDOW - 1, bool(flags[s : s + WINDOW].any()), bool(flags[s + WINDOW - 1]))
            for s in range(0, len(flags) - WINDOW + 1, STRIDE)]


wins = {name: windows(rec) for name, rec in c4.groupby("recording")}
summary = pd.DataFrame({
    "windows": {n: len(w) for n, w in wins.items()},
    "lying (any_lying)": {n: sum(x[2] for x in w) for n, w in wins.items()},
    "lying (last)": {n: sum(x[3] for x in w) for n, w in wins.items()},
})
summary.loc["total"] = summary.sum()
summary

## Watch a clip

`clip(recording, start, seconds)` plays the camera frames of one recording. The border is red on lying frames, and the strip underneath is the label timeline of the clip, with a marker for the current frame. Use the slider of the player to step through frames.

`start` is a frame position inside the recording, in seconds from its beginning. Videos are small (320 x 240) to keep the notebook light.

In [ ]:
def clip(recording, start_s=0.0, seconds=12.0, every=1, fps=5):
    """An inline video of `seconds` of one recording, starting `start_s` seconds in. `every=2` skips frames."""
    rec = frames_of(recording)
    first = int(round(start_s / 0.2))
    part = rec.iloc[first : first + int(round(seconds / 0.2)) : every]
    if part.empty:
        raise ValueError(f"{recording} has {len(rec)} frames, so there is nothing at {start_s} s")
    images = [im.convert("RGB").resize((320, 240)) for im in ds.select(part.index.tolist())["image"]]
    labels = part.label_pose.fillna("not labelled").tolist()
    codes = np.array([pose_code(None if l == "not labelled" else l) for l in labels])

    fig, (ax_img, ax_strip) = plt.subplots(2, 1, figsize=(4.4, 4.2), dpi=70,
                                           gridspec_kw={"height_ratios": [6, 1]})
    shown = ax_img.imshow(images[0])
    ax_img.set_xticks([]), ax_img.set_yticks([])
    title = ax_img.set_title("", fontsize=9)
    ax_strip.imshow(codes[None, :], aspect="auto", interpolation="nearest", cmap=ListedColormap(COLOURS), vmin=0, vmax=4)
    ax_strip.set_yticks([])
    ax_strip.set_xticks([0, len(codes) - 1], [f"{part.capture_idx.iloc[0] * 0.2:.0f} s", f"{part.capture_idx.iloc[-1] * 0.2:.0f} s"], fontsize=7)
    marker = ax_strip.axvline(0, color="black", lw=2)
    fig.tight_layout()

    def update(i):
        shown.set_data(images[i])
        lying = codes[i] == 3
        title.set_text(f"{labels[i]}   ({'LYING' if lying else 'not lying'})   position {part.capture_idx.iloc[i]}")
        title.set_color("#d7301f" if lying else "black")
        for spine in ax_img.spines.values():
            spine.set_edgecolor("#d7301f" if lying else "#cccccc")
            spine.set_linewidth(4 if lying else 1)
        marker.set_xdata([i, i])
        return shown, title, marker

    video = animation.FuncAnimation(fig, update, frames=len(images), interval=1000 / fps)
    plt.close(fig)
    return HTML(video.to_jshtml())


# the recording with the most lying frames, starting 4 s before its first lying run
busiest = overview.lying_frames.idxmax()
first_run = runs[busiest][0]
print(busiest, overview.person[busiest], "first lying run at", f"{first_run[0] * 0.2:.1f} s")
clip(busiest, start_s=max(0, first_run[0] * 0.2 - 4), seconds=12)

## Every lying episode as a strip of stills

One row per lying run: three frames from the 3 s before it, the first and the last lying frame, and three frames from the 3 s after it. The title of each still shows its pose label and its time relative to the start of the run. Lying frames have a red border. Change `MAX_EPISODES` to see more, or pass a recording name to `episodes` to look at one recording.

In [ ]:
def episodes(only=None, max_episodes=6, pad_s=3.0, n_around=3):
    """Stills around lying runs: `n_around` frames before, the first and last lying frame, `n_around` frames after.

    The first run of every recording comes first, so the rows show different people before any repeat.
    """
    todo = [(rec, k, run) for rec, rs in runs.items() if only in (None, rec) for k, run in enumerate(rs)]
    todo = sorted(todo, key=lambda item: (item[1], item[0]))[:max_episodes]
    if not todo:
        print("no lying runs")
        return
    n_frames = 2 * n_around + 2
    fig, axes = plt.subplots(len(todo), n_frames, figsize=(2.0 * n_frames, 1.8 * len(todo) + 0.3), squeeze=False)
    for row, (rec_name, _, (a, b)) in enumerate(todo):
        rec = frames_of(rec_name)
        pad = int(pad_s / 0.2)
        before = np.linspace(max(0, a - pad), max(0, a - 1), n_around).round().astype(int)
        after = np.linspace(min(len(rec) - 1, b + 1), min(len(rec) - 1, b + pad), n_around).round().astype(int)
        picks = list(before) + [a, b] + list(after)  # the run itself is always shown
        images = ds.select(rec.iloc[picks].index.tolist())["image"]
        for col, pick in enumerate(picks):
            ax = axes[row, col]
            ax.set_xticks([]), ax.set_yticks([])
            lying = rec.lying.iloc[pick]
            ax.imshow(images[col].convert("RGB").resize((320, 240)))
            ax.set_title(f"{rec.label_pose.iloc[pick]}  ({(pick - a) * 0.2:+.1f} s)", fontsize=7,
                         color="#d7301f" if lying else "black")
            for spine in ax.spines.values():
                spine.set_edgecolor("#d7301f" if lying else "#cccccc")
                spine.set_linewidth(3 if lying else 0.5)
        axes[row, 0].set_ylabel(f"{rec_name.split('__')[1]}\n{overview.person[rec_name]}\n{(b - a + 1) * 0.2:.1f} s lying",
                                fontsize=7, rotation=0, ha="right", va="center", labelpad=6)
    fig.tight_layout()
    plt.show()


MAX_EPISODES = 6
episodes(max_episodes=MAX_EPISODES)

## Your turn

Pick any recording and moment. `overview` lists the recording names, and the timeline above shows where lying happens.

```python
clip("2311__20231120_153152", start_s=60, seconds=15)   # a clip
episodes("2311__20231120_153152")                        # stills around each lying run of one recording
```